# Predicción de Gasto de Gas
Este notebook toma el histórico de consumo de gas (en $m^3$) de los últimos años y estima el gasto para los próximos 5 años (2027-2031), considerando la situación actual de los precios en Alemania (Alania).

In [1]:
%pip install pandas plotly matplotlib seaborn scikit-learn openpyxl
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Cargar los datos del Excel
url_excel = 'data/export_zaehlerstandshistorie_891288_20261003.xlsx'
df = pd.read_excel(url_excel)
df = df.rename(columns={'Unnamed: 0': 'Fecha', 'Unnamed: 4': 'Zaehlerstand', 'Unnamed: 5': 'Verbrauch'})
df = df[['Fecha', 'Verbrauch']]

# Convertir tipos
df['Fecha'] = pd.to_datetime(df['Fecha'], format='%d.%m.%Y', errors='coerce')
df['Verbrauch'] = pd.to_numeric(df['Verbrauch'], errors='coerce')
df = df.dropna()

df['Anio'] = df['Fecha'].dt.year

# Nos quedamos con los últimos 10 años
df = df[df['Anio'] >= (df['Anio'].max() - 10)]
df = df.sort_values('Anio').reset_index(drop=True)

df_anual = df.groupby('Anio')['Verbrauch'].sum().reset_index()
print(df_anual.shape)
df_anual.head()

(11, 2)


/Users/anaisabecolinaarismendi/Fritze Friends/.venv/lib/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Anio,Verbrauch
0,2016,7834.0
1,2017,8047.0
2,2018,8060.0
3,2019,7961.0
4,2020,7751.0


In [3]:
# Revisar valores nulos
print(df_anual.isna().sum())

Anio         0
Verbrauch    0
dtype: int64


In [4]:
# Graficar consumo histórico
fig = px.bar(df_anual, x="Anio", y="Verbrauch", title="Consumo Histórico de Gas (m3)")
fig.show()

In [5]:
# Preparar datos para Machine Learning
X = df_anual[['Anio']]
y = df_anual['Verbrauch']

# StandardScaler (similar al de Spotify)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
print("Entrenamiento:", X_train.shape, "Prueba:", X_test.shape)

Entrenamiento: (8, 1) Prueba: (3, 1)


In [6]:
# Entrenar varios modelos (Igual que en Spotify pero Regresores)
modelos = {
    "Regresión Lineal": LinearRegression(),
    "Árbol de Decisión": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42, n_estimators=100),
    "K-Vecinos": KNeighborsRegressor(n_neighbors=3)
}

resultados = []
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    preds = modelo.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    resultados.append({"Modelo": nombre, "MAE": mae})

df_resultados = pd.DataFrame(resultados).sort_values("MAE")
print(df_resultados)

              Modelo         MAE
2      Random Forest  626.070000
3          K-Vecinos  795.444444
1  Árbol de Decisión  832.000000
0   Regresión Lineal  888.758016


In [7]:
# Entrenamos el mejor modelo con todos los datos
mejor_modelo = RandomForestRegressor(random_state=42, n_estimators=100)
mejor_modelo.fit(X_scaled, y)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease o

In [8]:
# Predicción para los próximos 5 años (2027 a 2031)
anios_futuros = pd.DataFrame({'Anio': [2027, 2028, 2029, 2030, 2031]})
X_futuro_scaled = scaler.transform(anios_futuros[['Anio']])

consumo_predicho = mejor_modelo.predict(X_futuro_scaled)
anios_futuros['Consumo_m3_Estimado'] = consumo_predicho
anios_futuros

,Anio,Consumo_m3_Estimado
0,2027,6571.11
1,2028,6571.11
2,2029,6571.11
3,2030,6571.11
4,2031,6571.11


In [9]:
# Situación actual de gas en Alemania (octubre 2026): ~12 centavos de Euro por kWh
# 1 metro cúbico (m3) equivale aproximadamente a 10 kWh.
# Precio por m3 = 0.12 EUR * 10 = 1.20 EUR

precio_por_m3_eur = 1.20
anios_futuros['Costo_Estimado_EUR'] = anios_futuros['Consumo_m3_Estimado'] * precio_por_m3_eur

print("--- Gasto Estimado en Gas (Próximos 5 Años) ---")
print(anios_futuros)
print(f"Gasto Total Estimado (5 años): {anios_futuros['Costo_Estimado_EUR'].sum():.2f} €")

--- Gasto Estimado en Gas (Próximos 5 Años) ---
   Anio  Consumo_m3_Estimado  Costo_Estimado_EUR
0  2027              6571.11            7885.332
1  2028              6571.11            7885.332
2  2029              6571.11            7885.332
3  2030              6571.11            7885.332
4  2031              6571.11            7885.332
Gasto Total Estimado (5 años): 39426.66 €


In [10]:
# Graficar predicción
df_plot = pd.concat([
    df_anual.assign(Tipo='Histórico'),
    anios_futuros.rename(columns={'Consumo_m3_Estimado': 'Verbrauch'})[['Anio', 'Verbrauch']].assign(Tipo='Predicción')
])

fig = px.bar(df_plot, x="Anio", y="Verbrauch", color="Tipo", title="Consumo Histórico y Predicción de Gas")
fig.show()

In [11]:
# Guardar el modelo y el scaler
joblib.dump(mejor_modelo, 'modelo_gas.pkl')
joblib.dump(scaler, 'scaler_gas.pkl')
print("Modelo guardado exitosamente.")

Modelo guardado exitosamente.
